In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

base_results = Path(
    "/path/to/schemalens/fiben/results"
)

paths = {
    "sf1": base_results / "fiben_mongo_sf1" / "schemalens_reduction_analysis_hot.csv",
    "sf10": base_results / "fiben_mongo_sf10" / "schemalens_reduction_analysis_hot.csv",
    "sf30": base_results / "fiben_mongo_sf30" / "schemalens_reduction_analysis_hot.csv",
}

# ============================================================
# LOAD
# ============================================================

dfs = []

for scale, path in paths.items():
    df = pd.read_csv(path)
    df["scale_label"] = scale
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# ============================================================
# 1. SUMMARY (ATUALIZADO)
# ============================================================

summary_df = (
    all_df.groupby("scale_label")
    .agg(
        n_queries=("query_name", "count"),
        avg_DSR=("DSR", "mean"),
        top1_preservation=("top1_preserved_by_activated", "mean"),
        mean_activated_regret=("activated_regret", "mean"),
        mean_primary_regret=("primary_regret", "mean"),
        mean_near_best_ratio=("near_best_ratio", "mean"),

        primary_winners=("best_group", lambda x: (x == "primary").sum()),
        secondary_winners=("best_group", lambda x: (x == "secondary_affected").sum()),
        control_winners=("best_group", lambda x: (x == "control").sum()),
    )
    .reset_index()
)

display(summary_df)

# ============================================================
# 2. P95
# ============================================================

pivot_p95 = all_df.pivot_table(
    index=["official_id", "query_name"],
    columns="scale_label",
    values="best_p95_ms"
).reset_index()

pivot_p95["delta_sf10_vs_sf1"] = (
    pivot_p95["sf10"] - pivot_p95["sf1"]
) / pivot_p95["sf1"]

pivot_p95["delta_sf30_vs_sf1"] = (
    pivot_p95["sf30"] - pivot_p95["sf1"]
) / pivot_p95["sf1"]

display(pivot_p95)

# ============================================================
# 3. STABILITY
# ============================================================

pivot_config = all_df.pivot_table(
    index=["official_id", "query_name"],
    columns="scale_label",
    values="best_config",
    aggfunc="first"
).reset_index()

pivot_config["stable_across_scales"] = pivot_config.apply(
    lambda row: len(set([row["sf1"], row["sf10"], row["sf30"]])) == 1,
    axis=1
)

display(pivot_config)

print("Stable ratio:", pivot_config["stable_across_scales"].mean())

# ============================================================
# 4. NEAR-BEST COMPARISON
# ============================================================

pivot_near = all_df.pivot_table(
    index=["official_id", "query_name"],
    columns="scale_label",
    values="near_best_ratio"
).reset_index()

display(pivot_near)

# ============================================================
# SAVE
# ============================================================

output_dir = base_results / "cross_scale_analysis_fiben"
output_dir.mkdir(exist_ok=True)

summary_df.to_csv(output_dir / "cross_scale_summary.csv", index=False)
pivot_p95.to_csv(output_dir / "cross_scale_p95.csv", index=False)
pivot_config.to_csv(output_dir / "cross_scale_configs.csv", index=False)
pivot_near.to_csv(output_dir / "cross_scale_near_best.csv", index=False)

print("Saved to:", output_dir)